# DeepDetect Experiments V2: dino_vits8
V2 pipeline: v5 skin-tone patch + phase FFT + original cleaner + dual transforms.
Spatial branch: full degradation augmentations.
Frequency branch: spatial aug only — patch selected from clean image.

Best hyperparameters from Optuna tuning (trial 13):
  backbone_lr=4.85e-05, lr=2.79e-04, freq_aux_weight=0.500,
  diversity_weight=0.058, gate_init_bias=0.215


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
VRAM: 25.2 GB


## Best Hyperparameters

In [2]:
BEST_BACKBONE_LR       = 4.85e-05
BEST_LR                = 2.79e-04
BEST_FREQ_AUX_WEIGHT   = 0.500
BEST_DIVERSITY_WEIGHT  = 0.058
BEST_GATE_INIT_BIAS    = 0.215


## Shared Config

In [3]:
from config import Config
from data.deepdetect_dual import get_deepdetect_dual_loaders
from experiments.train import train_v2
from experiments.evaluate import full_evaluation_v2
from experiments.baseline_spatial_only import run_spatial_only_baseline

BACKBONE = "dino_vits8"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.deepdetect_root      = "../data/raw/deep_detect/data"
    cfg.data.dataset              = "deepdetect"
    cfg.data.image_size           = 224
    cfg.data.batch_size           = 88
    cfg.data.num_workers          = 4
    cfg.backbone.name             = BACKBONE
    cfg.backbone.pretrained       = True
    cfg.backbone.img_size         = 224
    cfg.backbone.frozen           = frozen
    cfg.fusion.mode               = fusion_mode
    cfg.frequency.use_v2_pipeline = True
    cfg.frequency.cleaner_filters = 3
    cfg.frequency.patch_size      = 56
    cfg.train.early_stopping_patience = 30  
    cfg.train.epochs              = 35
    cfg.train.backbone_lr         = BEST_BACKBONE_LR
    cfg.train.lr                  = BEST_LR
    cfg.loss.freq_aux_weight      = BEST_FREQ_AUX_WEIGHT
    cfg.fusion.diversity_weight   = BEST_DIVERSITY_WEIGHT
    cfg.fusion.gate_init_bias     = BEST_GATE_INIT_BIAS
    # Spatial branch — full augmentations
    cfg.data.jpeg_aug             = True
    cfg.data.blur_aug             = True
    cfg.data.noise_aug            = True
    cfg.data.recompression_aug    = True
    cfg.data.mixed_aug            = True
    cfg.data.mixed_aug_prob       = 0.3
    cfg.experiment_name = f"dd_v2_{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes           = f"DeepDetect V2 · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

cfg_data = make_cfg("joint_only")
train_loader, val_loader, test_loader = get_deepdetect_dual_loaders(cfg_data)


Train: 76,848  Val: 13,561  Test: 21,776


## Experiment 0: Spatial-Only Baseline

In [4]:
cfg0 = make_cfg("joint_only")
cfg0.experiment_name = f"dd_v2_{BACKBONE}_spatial_only"
cfg0.notes           = f"DeepDetect V2 spatial-only baseline"
cfg0.train.epochs    = 20
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")


Device: cuda
Training spatial-only baseline (dino_vits8) for 20 epochs...
Train: 76,848  Val: 13,561


Epoch   1/20 | train_loss=0.2489 | val_acc=94.0%


Epoch   2/20 | train_loss=0.1187 | val_acc=97.9%


Epoch   3/20 | train_loss=0.0927 | val_acc=92.6%


Epoch   4/20 | train_loss=0.0797 | val_acc=96.7%


...

Epoch  16/20 | train_loss=0.0032 | val_acc=99.3%


Epoch  17/20 | train_loss=0.0023 | val_acc=99.2%


Epoch  18/20 | train_loss=0.0012 | val_acc=99.5%


Epoch  19/20 | train_loss=0.0008 | val_acc=99.5%


Epoch  20/20 | train_loss=0.0009 | val_acc=99.6%

Spatial-only results (dino_vits8):
  Accuracy: 91.8%
  AUC-ROC:  0.962
  F1:       0.920
Results saved → ./results/results.csv  (dd_v2_dino_vits8_spatial_only, acc=0.918)

Spatial-only floor: 91.8%


## Experiment 1: Joint-Only

In [5]:
cfg1 = make_cfg("joint_only")
train_v2(cfg1, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_dino_vits8_joint_only [V2 pipeline]
Backbone: dino_vits8 | Fusion: joint_only | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5863 | val_acc=94.7% | val_auc=0.999 | val_f1=0.938 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=94.7%)


Epoch   2/35 | train_loss=0.4359 | val_acc=99.5% | val_auc=1.000 | val_f1=0.994 | best=94.7% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=99.5%)


Epoch   3/35 | train_loss=0.4034 | val_acc=98.5% | val_auc=1.000 | val_f1=0.984 | best=99.5% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=98.5%)


Epoch   4/35 | train_loss=0.3829 | val_acc=99.1% | val_auc=1.000 | val_f1=0.990 | best=99.5% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=99.1%)


...

Epoch  32/35 | train_loss=0.2654 | val_acc=99.7% | val_auc=1.000 | val_f1=0.996 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.7%)


Epoch  33/35 | train_loss=0.2645 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.7%)


Epoch  34/35 | train_loss=0.2651 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.7%)


Epoch  35/35 | train_loss=0.2649 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.7%)

Training complete. Best val accuracy: 99.8%
Results will be logged to CSV after full_evaluation_v2().


0.9977140328884301

In [6]:
results1 = full_evaluation_v2(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)

Loaded checkpoint: ./checkpoints/best_dd_v2_dino_vits8_joint_only.pt

EVALUATION — dd_v2_dino_vits8_joint_only
Backbone: dino_vits8 | Fusion: joint_only | Frozen: False
  Joint accuracy:     94.4%
  AUC-ROC:            0.980
  F1:                 0.944
  Spatial-only:       94.3%
  Freq-only:          70.2%
  Delta (Δ):          +0.1%  (freq branch contribution)

  No warning signs triggered.
Results saved → ./results/results.csv  (dd_v2_dino_vits8_joint_only, acc=0.9439)



## Experiment 2: Scalar Fusion

In [7]:
cfg2 = make_cfg("scalar")
train_v2(cfg2, train_loader, val_loader, test_loader)

Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_dino_vits8_scalar [V2 pipeline]
Backbone: dino_vits8 | Fusion: scalar | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5600 | val_acc=98.9% | val_auc=0.999 | val_f1=0.988 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=98.9%)


Epoch   2/35 | train_loss=0.4275 | val_acc=99.2% | val_auc=1.000 | val_f1=0.992 | best=98.9% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=99.2%)


Epoch   3/35 | train_loss=0.3981 | val_acc=99.2% | val_auc=1.000 | val_f1=0.991 | best=99.2% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=99.2%)


Epoch   4/35 | train_loss=0.3825 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992 | best=99.2% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=99.3%)


...

Epoch  32/35 | train_loss=0.2630 | val_acc=99.7% | val_auc=1.000 | val_f1=0.996 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.7%)


Epoch  33/35 | train_loss=0.2626 | val_acc=99.6% | val_auc=1.000 | val_f1=0.996 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.6%)


Epoch  34/35 | train_loss=0.2629 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.7%)


Epoch  35/35 | train_loss=0.2608 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.7%)

Training complete. Best val accuracy: 99.8%
Results will be logged to CSV after full_evaluation_v2().


0.9982302190103974

In [8]:
results2 = full_evaluation_v2(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)

Loaded checkpoint: ./checkpoints/best_dd_v2_dino_vits8_scalar.pt

EVALUATION — dd_v2_dino_vits8_scalar
Backbone: dino_vits8 | Fusion: scalar | Frozen: False
  Joint accuracy:     96.6%
  AUC-ROC:            0.992
  F1:                 0.965
  Spatial-only:       96.6%
  Freq-only:          73.0%
  Delta (Δ):          +0.0%  (freq branch contribution)
Results saved → ./results/results.csv  (dd_v2_dino_vits8_scalar, acc=0.9658)



## Experiment 3: Gating Fusion

In [9]:
cfg3 = make_cfg("gating")
train_v2(cfg3, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell


Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_dino_vits8_gating [V2 pipeline]
Backbone: dino_vits8 | Fusion: gating | Frozen: False
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.5846 | val_acc=98.7% | val_auc=0.999 | val_f1=0.986 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=98.7%)


Epoch   2/35 | train_loss=0.4575 | val_acc=99.6% | val_auc=1.000 | val_f1=0.995 | best=98.7% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=99.6%)


Epoch   3/35 | train_loss=0.4333 | val_acc=99.2% | val_auc=1.000 | val_f1=0.992 | best=99.6% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=99.2%)


...

Epoch  32/35 | train_loss=0.2862 | val_acc=99.8% | val_auc=1.000 | val_f1=0.997 | best=99.7% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=99.8%)


Epoch  33/35 | train_loss=0.2870 | val_acc=99.8% | val_auc=1.000 | val_f1=0.998 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=99.8%)


Epoch  34/35 | train_loss=0.2860 | val_acc=99.8% | val_auc=1.000 | val_f1=0.998 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=99.8%)


Epoch  35/35 | train_loss=0.2873 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997 | best=99.8% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=99.7%)

Training complete. Best val accuracy: 99.8%
Results will be logged to CSV after full_evaluation_v2().


0.9978615146375636

In [10]:
results3 = full_evaluation_v2(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_v2_dino_vits8_gating.pt

EVALUATION — dd_v2_dino_vits8_gating
Backbone: dino_vits8 | Fusion: gating | Frozen: False
  Joint accuracy:     96.2%
  AUC-ROC:            0.984
  F1:                 0.962
  Spatial-only:       96.2%
  Freq-only:          70.4%
  Delta (Δ):          -0.0%  (freq branch contribution)

  Gate distribution:
    entropy:  1.579 nats  (OK)
    mean:     0.561
    variance: 0.0035
Results saved → ./results/results.csv  (dd_v2_dino_vits8_gating, acc=0.962)



## Experiment 4: Gating Frozen

In [11]:
cfg4 = make_cfg("gating", frozen=True)
train_v2(cfg4, train_loader, val_loader, test_loader)


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
Using differential lr: backbone=4.85e-05, others=2.79e-04

Experiment: dd_v2_dino_vits8_gating_frozen [V2 pipeline]
Backbone: dino_vits8 | Fusion: gating | Frozen: True
Epochs: 35 | LR: 0.000279 | Batch: 88
Spatial branch: full aug | Frequency branch: spatial aug only



Epoch   1/35 | train_loss=0.8459 | val_acc=90.2% | val_auc=0.977 | val_f1=0.885 | best=0.0% | patience=0/30
  -> Saved checkpoint (epoch 1, val_acc=90.2%)


Epoch   2/35 | train_loss=0.7279 | val_acc=90.3% | val_auc=0.980 | val_f1=0.886 | best=90.2% | patience=0/30
  -> Saved checkpoint (epoch 2, val_acc=90.3%)


Epoch   3/35 | train_loss=0.6750 | val_acc=92.6% | val_auc=0.986 | val_f1=0.916 | best=90.3% | patience=0/30
  -> Saved checkpoint (epoch 3, val_acc=92.6%)


Epoch   4/35 | train_loss=0.6497 | val_acc=94.2% | val_auc=0.987 | val_f1=0.936 | best=92.6% | patience=0/30
  -> Saved checkpoint (epoch 4, val_acc=94.2%)


...

Epoch  32/35 | train_loss=0.4395 | val_acc=95.1% | val_auc=0.995 | val_f1=0.945 | best=95.7% | patience=0/30
  -> Saved checkpoint (epoch 32, val_acc=95.1%)


Epoch  33/35 | train_loss=0.4380 | val_acc=94.9% | val_auc=0.995 | val_f1=0.942 | best=95.7% | patience=0/30
  -> Saved checkpoint (epoch 33, val_acc=94.9%)


Epoch  34/35 | train_loss=0.4323 | val_acc=95.1% | val_auc=0.995 | val_f1=0.945 | best=95.7% | patience=0/30
  -> Saved checkpoint (epoch 34, val_acc=95.1%)


Epoch  35/35 | train_loss=0.4379 | val_acc=95.1% | val_auc=0.995 | val_f1=0.945 | best=95.7% | patience=0/30
  -> Saved checkpoint (epoch 35, val_acc=95.1%)

Training complete. Best val accuracy: 95.7%
Results will be logged to CSV after full_evaluation_v2().


0.9565666248801711

In [12]:
results4 = full_evaluation_v2(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_v2_dino_vits8_gating_frozen.pt

EVALUATION — dd_v2_dino_vits8_gating_frozen
Backbone: dino_vits8 | Fusion: gating | Frozen: True
  Joint accuracy:     89.7%
  AUC-ROC:            0.947
  F1:                 0.893
  Spatial-only:       79.0%
  Freq-only:          71.5%
  Delta (Δ):          +10.6%  (freq branch contribution)

  Gate distribution:
    entropy:  2.884 nats  (OK)
    mean:     0.610
    variance: 0.0835

  No warning signs triggered.
Results saved → ./results/results.csv  (dd_v2_dino_vits8_gating_frozen, acc=0.8965)



## Summary

In [13]:
import pandas as pd
df = pd.read_csv("./results/results.csv",  on_bad_lines='skip')
mask = df["experiment_name"].str.startswith(f"dd_v2_{BACKBONE}")
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))


               experiment_name     fusion  frozen  accuracy  auc_roc     f1  gate_entropy
 dd_v2_dino_vits8_spatial_only joint_only   False    0.9180   0.9619 0.9204        0.0000
   dd_v2_dino_vits8_joint_only joint_only   False    0.9439   0.9797 0.9442        0.0000
       dd_v2_dino_vits8_scalar     scalar   False    0.9658   0.9918 0.9653        0.0000
       dd_v2_dino_vits8_gating     gating   False    0.9620   0.9835 0.9616        1.5786
dd_v2_dino_vits8_gating_frozen     gating    True    0.8965   0.9466 0.8930        2.8841
